# Credit Card Fraud Detection
## IBM3201 Data Mining and Predictive Analytics
**Author:** Suruthi Kattampalayam Sivasankar · INTI International University · January 2026

This notebook follows the **CRISP-DM** framework to build a binary classifier that identifies fraudulent credit card transactions from a heavily imbalanced real-world dataset (284,807 transactions, 0.17% fraud).

**Pipeline overview:**
1. Data loading \& quality checks
2. Exploratory Data Analysis (EDA)
3. Preprocessing (scaling, SMOTE / class weights)
4. Model training — Random Forest (two strategies compared)
5. Evaluation — Confusion Matrix, ROC-AUC, Feature Importance

**Dataset:** Provided as course material — download from [Google Drive](https://drive.google.com/drive/folders/1cO0ckFPA0BIxY3Gx7fJ54WS-EymvnmNM?usp=sharing)  
Place both CSVs in the same directory as this notebook before running.

## 1. Setup — Install Dependencies & Import Libraries

In [ ]:
# Install imbalanced-learn if not already present
!pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
)
from imblearn.over_sampling import SMOTE

# Plotting defaults
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
print('✓ All libraries loaded successfully.')

## 2. Data Loading

> **To run locally:** place `creditcard_transactions_part1.csv` and `creditcard_transactions_part2.csv` in the same directory as this notebook.  
> **On Google Colab:** uncomment the upload block below.

In [ ]:
# ── Colab file upload (uncomment if running on Google Colab) ──────────────────
# from google.colab import files
# uploaded = files.upload()  # select BOTH CSV files
# ─────────────────────────────────────────────────────────────────────────────

df1 = pd.read_csv('../data/creditcard_transactions_part1.csv')
df2 = pd.read_csv('../data/creditcard_transactions_part2.csv')

df = pd.concat([df1, df2], ignore_index=True)

print(f'Combined dataset shape : {df.shape}')
print(f'Columns                : {list(df.columns)}')
df.head(3)

## 3. Data Quality Check

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum().sum()
print(f'Total missing values : {missing}')

# ── Duplicate rows ────────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'Duplicate rows found : {n_dupes}')

df = df.drop_duplicates()
print(f'Clean dataset shape  : {df.shape}')

In [ ]:
# ── Descriptive statistics for key features ───────────────────────────────────
key_features = ['Time', 'Amount', 'V14', 'V17', 'V12', 'Class']
df[key_features].describe().round(4)

In [ ]:
# ── Target variable distribution ──────────────────────────────────────────────
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Legitimate (0) : {class_counts[0]:,}  ({class_pct[0]:.4f}%)')
print(f'  Fraud      (1) : {class_counts[1]:,}  ({class_pct[1]:.4f}%)')

## 4. Exploratory Data Analysis

In [ ]:
# ── Figure 1: Class Distribution ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Legitimate (0)', 'Fraud (1)'],
              [class_counts[0], class_counts[1]],
              color=['steelblue', 'tomato'])
for bar, count in zip(bars, [class_counts[0], class_counts[1]]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2000,
            f'{count:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Class Distribution (Legitimate vs Fraud)', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../outputs/charts/class_distribution.png')
plt.show()

In [ ]:
# ── Figure 2: Transaction Amount Distribution by Class ────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df[df['Class'] == 0]['Amount'].clip(upper=1000).plot(
    kind='hist', bins=50, density=True, alpha=0.6, color='steelblue',
    label='Legitimate', ax=ax)
df[df['Class'] == 1]['Amount'].clip(upper=1000).plot(
    kind='hist', bins=50, density=True, alpha=0.6, color='tomato',
    label='Fraud', ax=ax)
ax.set_xlabel('Transaction Amount ($)')
ax.set_ylabel('Density')
ax.set_title('Transaction Amount Distribution by Class (capped at $1,000)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/charts/amount_distribution.png')
plt.show()

In [ ]:
# ── Figure 3: Transaction Time Distribution by Class ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df[df['Class'] == 0]['Time'].plot(
    kind='hist', bins=48, density=True, alpha=0.6, color='steelblue',
    label='Legitimate', ax=ax)
df[df['Class'] == 1]['Time'].plot(
    kind='hist', bins=48, density=True, alpha=0.6, color='tomato',
    label='Fraud', ax=ax)
ax.set_xlabel('Time (seconds from first transaction)')
ax.set_ylabel('Density')
ax.set_title('Transaction Time Distribution by Class')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/charts/time_distribution.png')
plt.show()

In [ ]:
# ── Figure 4: Boxplots — Key PCA Features by Class ───────────────────────────
pca_features = ['V14', 'V17', 'V12']
plot_data = []
labels = []
for feat in pca_features:
    for cls, name in [(0, 'Legit'), (1, 'Fraud')]:
        plot_data.append(df[df['Class'] == cls][feat].values)
        labels.append(f'{feat}\n{name}')

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(plot_data, labels=labels, patch_artist=True)
colors = ['steelblue', 'tomato'] * len(pca_features)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title('Key PCA Feature Distributions by Class')
ax.set_ylabel('Feature Value')
plt.tight_layout()
plt.savefig('../outputs/charts/pca_boxplots.png')
plt.show()

In [ ]:
# ── Figure 5: Correlation Heatmap ─────────────────────────────────────────────
top_feats = ['V14', 'V17', 'V12', 'V10', 'V11', 'V16', 'V3', 'V7', 'Amount', 'Class']
corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Top Features vs Class')
plt.tight_layout()
plt.savefig('../outputs/charts/correlation_heatmap.png')
plt.show()

In [ ]:
# ── Figure 6: Scatter Plot — Amount vs V14 by Class ──────────────────────────
legit_sample = df[df['Class'] == 0].sample(1000, random_state=RANDOM_STATE)
fraud_all    = df[df['Class'] == 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(legit_sample['Amount'], legit_sample['V14'],
           alpha=0.4, s=15, color='steelblue', label='Legitimate (sample)')
ax.scatter(fraud_all['Amount'], fraud_all['V14'],
           alpha=0.7, s=20, color='tomato', label='Fraud (all)')
ax.set_xlabel('Transaction Amount ($)')
ax.set_ylabel('V14 (PCA Component)')
ax.set_title('Transaction Amount vs V14 by Class')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/charts/scatter_amount_v14.png')
plt.show()

In [ ]:
# ── Cross-tabulation: Fraud Rate by Transaction Amount Bucket ─────────────────
bins   = [0, 50, 200, 500, 1000, df['Amount'].max() + 1]
labels = ['$0–$50', '$50–$200', '$200–$500', '$500–$1,000', '$1,000+']
df['AmountBucket'] = pd.cut(df['Amount'], bins=bins, labels=labels, right=False)

cross_tab = (df.groupby('AmountBucket', observed=True)['Class']
               .agg(Legitimate=lambda x: (x == 0).sum(),
                    Fraud=lambda x: (x == 1).sum(),
                    Total='count')
               .assign(FraudRate=lambda d: (d['Fraud'] / d['Total'] * 100).round(4)))
print(cross_tab.to_string())

## 5. Data Preprocessing

In [ ]:
# ── Feature matrix and target vector ──────────────────────────────────────────
feature_cols = [c for c in df.columns if c not in ['Class', 'AmountBucket']]
X = df[feature_cols].copy()
y = df['Class'].copy()

# ── Normalise Amount and Time (V1–V28 are already PCA-standardised) ───────────
scaler = StandardScaler()
X[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

# ── Stratified 80/20 train-test split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f'Training set  : {X_train.shape[0]:,} records')
print(f'Test set      : {X_test.shape[0]:,} records')
print(f'Fraud in test : {y_test.sum()} ({y_test.mean()*100:.4f}%)')

In [ ]:
# ── Apply SMOTE on training data only ─────────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — training set size : {X_train_smote.shape[0]:,}')
print(f'Class balance (SMOTE)           : {pd.Series(y_train_smote).value_counts().to_dict()}')

## 6. Model Training

Two strategies for handling class imbalance are trained and compared:

| | Model A | Model B |
|---|---|---|
| **Strategy** | SMOTE oversampling | Class weight adjustment |
| **Data modified?** | Yes (training only) | No |
| **Algorithm** | Random Forest (100 trees) | Random Forest (100 trees) |

In [ ]:
# ── Model A: Random Forest + SMOTE ───────────────────────────────────────────
rf_smote = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote  = rf_smote.predict(X_test)
y_prob_smote  = rf_smote.predict_proba(X_test)[:, 1]

print('=== Model A — Random Forest + SMOTE ===')
print(classification_report(y_test, y_pred_smote,
                             target_names=['Legitimate', 'Fraud']))

In [ ]:
# ── Model B: Random Forest + Class Weights ────────────────────────────────────
rf_weights = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1)
rf_weights.fit(X_train, y_train)

y_pred_weights = rf_weights.predict(X_test)
y_prob_weights = rf_weights.predict_proba(X_test)[:, 1]

print('=== Model B — Random Forest + Class Weights ===')
print(classification_report(y_test, y_pred_weights,
                             target_names=['Legitimate', 'Fraud']))

## 7. Evaluation

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title, cmap in zip(
    axes,
    [y_pred_smote, y_pred_weights],
    ['Model A — SMOTE', 'Model B — Class Weights'],
    ['Blues', 'Oranges'],
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(title, fontsize=12)

plt.suptitle('Confusion Matrices — Test Set (56,746 transactions)', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/charts/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
auc_smote   = roc_auc_score(y_test, y_prob_smote)
auc_weights = roc_auc_score(y_test, y_prob_weights)

fpr_s, tpr_s, _ = roc_curve(y_test, y_prob_smote)
fpr_w, tpr_w, _ = roc_curve(y_test, y_prob_weights)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_s, tpr_s, color='tomato',    lw=2, label=f'SMOTE        (AUC = {auc_smote:.4f})')
ax.plot(fpr_w, tpr_w, color='steelblue', lw=2, label=f'Class Weights (AUC = {auc_weights:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison — SMOTE vs Class Weights')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/charts/roc_curves.png')
plt.show()

print(f'ROC-AUC  SMOTE        : {auc_smote:.4f}')
print(f'ROC-AUC  Class Weights: {auc_weights:.4f}')

In [ ]:
# ── Feature Importance — Model A (SMOTE) ──────────────────────────────────────
importances = pd.Series(
    rf_smote.feature_importances_, index=X.columns
).sort_values(ascending=False)

top15 = importances.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top15[::-1].plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Feature Importances — Random Forest (SMOTE)')
plt.tight_layout()
plt.savefig('../outputs/charts/feature_importance.png')
plt.show()

print('\nTop 10 Feature Importances:')
print(top15.head(10).round(4).to_string())

In [ ]:
# ── Model Comparison Summary Table ────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

summary = pd.DataFrame({
    'Model A — SMOTE': {
        'Fraud Precision': precision_score(y_test, y_pred_smote),
        'Fraud Recall'   : recall_score(y_test, y_pred_smote),
        'Fraud F1-Score' : f1_score(y_test, y_pred_smote),
        'ROC-AUC'        : auc_smote,
        'Accuracy'       : accuracy_score(y_test, y_pred_smote),
    },
    'Model B — Class Weights': {
        'Fraud Precision': precision_score(y_test, y_pred_weights),
        'Fraud Recall'   : recall_score(y_test, y_pred_weights),
        'Fraud F1-Score' : f1_score(y_test, y_pred_weights),
        'ROC-AUC'        : auc_weights,
        'Accuracy'       : accuracy_score(y_test, y_pred_weights),
    },
}).round(4)

print(summary.to_string())

## 8. Conclusion

**Model A (Random Forest + SMOTE)** is recommended for production:

- **Higher recall** — detects 3 more fraud cases per 56,746 transactions vs Model B. Scaled to millions of daily transactions this represents hundreds of additional frauds intercepted.
- **Higher ROC-AUC (0.9436 vs 0.9141)** — better generalisation and more flexibility to tune the classification threshold to a bank's specific risk appetite.
- **Domain priority** — in fraud detection, the cost of a missed fraud (financial loss, regulatory risk) far outweighs the cost of a false alarm.

**Model B (Class Weights)** remains a strong alternative where minimising false positives and protecting customer experience is the primary concern (only 1 false positive vs 6 for Model A).

### Key findings
- V14 alone accounts for ~22.8% of the model's decision-making.
- V14 + V10 + V17 together cover ~44.5% of total feature importance.
- Fraud rate increases proportionally with transaction amount (highest at 0.42% in the $500–$1,000 bucket).
- Fraudulent transactions show no human behavioural rhythm — they spike at all hours, including low-activity periods.

### Future work
- Engineer time-of-day, transaction velocity, and amount-deviation-ratio features.
- Compare against XGBoost / LightGBM and stacking ensembles.
- Add SHAP explainability for regulatory compliance.
- Implement continuous model retraining on a rolling data window.